# Inference Optimization Techniques

## KV-Cache Optimization

During autoregressive generation, the key-value (KV) cache stores previously computed attention keys and values to avoid recomputation. Without optimization, KV-cache memory grows linearly with sequence length: $\text{Memory} = 2 \times \text{batch\_size} \times \text{seq\_len} \times \text{hidden\_dim} \times \text{bytes\_per\_value}$. Techniques like multi-query attention (MQA) and grouped-query attention (GQA) reduce cache size by sharing keys/values across attention heads.

```python title="example1.py"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model with optimized KV-cache
model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/gpt-neo-125M",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

# Generate with KV-cache (automatic in transformers)
prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# use_cache=True enables KV-cache by default
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    use_cache=True,  # Enable KV-cache
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/deep-learning/mod-27.ipynb)

```
The future of AI is bright and full of possibilities. Machine learning continues to advance...
```

## Flash Attention

Flash Attention reduces memory I/O by computing attention in blocks without materializing the full attention matrix. Standard attention: $O = \text{softmax}(QK^T/\sqrt{d})V$ requires $O(N^2)$ memory. Flash Attention computes attention in tiles, reducing memory bandwidth by 10-100x while maintaining numerical accuracy.

```python title="example2.py"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from flash_attn import flash_attn_func

# Load model with Flash Attention support
model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/gpt-neo-125M",
    attn_implementation="flash_attention_2",  # Enable Flash Attention
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

# Generate with Flash Attention
prompt = "Explain quantum computing in simple terms:"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Flash Attention is used automatically in forward pass
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    use_cache=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

> **💡 Tip:** Flash Attention requires CUDA-capable GPUs. For CPU inference, use standard attention or quantized models.

## Quantized Inference

Running quantized models (INT8, INT4) reduces memory and speeds up computation. Combined with KV-cache quantization, this enables serving large models on consumer hardware.

```python title="example3.py"
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    "EleutherAI/gpt-neo-125M",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

# Inference with quantized model
prompt = "What is machine learning?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    use_cache=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary benefit of KV-cache in inference?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300001" value="0">
      <span>Increases model accuracy</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300001" value="1">
      <span>Avoids recomputing attention for previous tokens</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300001" value="2">
      <span>Reduces model parameters</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300001" value="3">
      <span>Improves tokenization</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What memory complexity does Flash Attention reduce from?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300002" value="0">
      <span>O(N) to O(log N)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300002" value="1">
      <span>O(N³) to O(N²)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300002" value="2">
      <span>O(N²) to O(N) via block-wise computation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4387300002" value="3">
      <span>O(N) to O(1)</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>